# ML Enhancement: Market Basket Analysis (pair co-occurrence)

Discover products frequently purchased together by counting product pairs over
every `SALE` basket.

## Business Value
- Cross-sell recommendations ("Customers also bought")
- Optimize store layouts based on co-purchase patterns
- Bundle pricing opportunities
- Promotion targeting with complementary products

## Method
- A basket is one `SALE` receipt's distinct products; single-item baskets
  count too, so support and lift are relative to all baskets.
- Support = baskets with both products / all baskets; confidence =
  P(consequent | antecedent); lift = confidence / P(consequent).
- A rule needs at least `MIN_PAIR_BASKETS` co-purchases, `MIN_CONFIDENCE` and
  a lift above `MIN_LIFT`, so rare noise pairs don't top the ranking. The
  generator's complementary products (`baskets.with_complements`) are the
  structure these rules recover.
- Above `MAX_PAIR_BASKETS` baskets, a deterministic hash sample of baskets
  bounds the pair count; support, confidence and lift are ratios over the
  sample.
- When no pair meets the thresholds (small datasets), pairs bought together in
  at least `FALLBACK_MIN_PAIR_BASKETS` baskets are published instead
  (`rule_source = cooccurrence_fallback`).

## Output
- Association rules with support, confidence, and lift are saved to the configured gold table
- `computed_at` is the source as-of time (the latest Silver receipt)

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.utils import AnalysisException
from datetime import datetime, timezone
import os
import mlflow


In [ ]:
# =============================================================================
# PARAMETERS
# =============================================================================

def get_env(var_name, default=None):
    return os.environ.get(var_name, default)

LAKEHOUSE_NAME = get_env("LAKEHOUSE_NAME", default="retail_lakehouse")
SILVER_DB = get_env("SILVER_DB", default="ag")
GOLD_DB = get_env("GOLD_DB", default="au")
EXPERIMENT_NAME = get_env("MLFLOW_EXPERIMENT", default="market_basket")
RECEIPTS_TABLE = get_env("RECEIPTS_TABLE", default="fact_receipts")
RECEIPT_LINES_TABLE = get_env("RECEIPT_LINES_TABLE", default="fact_receipt_lines")
PRODUCT_ASSOCIATIONS_TABLE = get_env("PRODUCT_ASSOCIATIONS_TABLE", default="product_associations")
PRODUCT_RECOMMENDATIONS_TABLE = get_env("PRODUCT_RECOMMENDATIONS_TABLE", default="product_recommendations")
PRODUCT_ASSOCIATIONS_TABLE_NAME = f"{LAKEHOUSE_NAME}.{GOLD_DB}.{PRODUCT_ASSOCIATIONS_TABLE}"
PRODUCT_RECOMMENDATIONS_TABLE_NAME = f"{LAKEHOUSE_NAME}.{GOLD_DB}.{PRODUCT_RECOMMENDATIONS_TABLE}"
RECEIPT_TYPE_FILTER = get_env("RECEIPT_TYPE_FILTER", default="SALE")

# Rule thresholds. Support is over all SALE baskets; a minimum co-purchase
# count keeps rare noise pairs out of the lift ranking.
MIN_PAIR_BASKETS = int(get_env("MIN_PAIR_BASKETS", default="20"))
MIN_SUPPORT = float(get_env("MIN_SUPPORT", default="0.0"))
MIN_CONFIDENCE = float(get_env("MIN_CONFIDENCE", default="0.05"))
MIN_LIFT = float(get_env("MIN_LIFT", default="1.0"))
MAX_RULES_PER_PRODUCT = int(get_env("MAX_RULES_PER_PRODUCT", default="5"))
TOP_N_RULES = int(get_env("TOP_N_RULES", default="500"))
# Above this many baskets, pairs are counted on a deterministic hash sample.
MAX_PAIR_BASKETS = int(get_env("MAX_PAIR_BASKETS", default="2000000"))
# Fallback when no pair meets the thresholds: small-basket store types (luxury
# averages 1.6 items) rarely repeat a pair often enough.
FALLBACK_MIN_PAIR_BASKETS = int(get_env("FALLBACK_MIN_PAIR_BASKETS", default="2"))



print(f"Configuration:")
print(f"  SILVER_DB={SILVER_DB}, GOLD_DB={GOLD_DB}")
print(f"  Source tables: {RECEIPTS_TABLE}, {RECEIPT_LINES_TABLE}")
print(f"  Receipt type filter: {RECEIPT_TYPE_FILTER}")
print(f"  Output tables: {PRODUCT_ASSOCIATIONS_TABLE_NAME}, {PRODUCT_RECOMMENDATIONS_TABLE_NAME}")
print(
    f"  MIN_PAIR_BASKETS={MIN_PAIR_BASKETS}, MIN_SUPPORT={MIN_SUPPORT}, "
    f"MIN_CONFIDENCE={MIN_CONFIDENCE}, MIN_LIFT={MIN_LIFT}"
)
print(f"  MAX_RULES_PER_PRODUCT={MAX_RULES_PER_PRODUCT}, TOP_N_RULES={TOP_N_RULES}")

# Physical ML contract used by repository validation and the pre-write gate.
ML_SCHEMA_VERSION = "1.0"
ML_SOURCE_TABLES = ('fact_receipts', 'fact_receipt_lines')
ML_OUTPUT_CONTRACTS = {'product_associations': [('antecedent', 'array<long>', False),
                          ('consequent', 'array<long>', False),
                          ('support', 'double', False),
                          ('confidence', 'double', False),
                          ('lift', 'double', False),
                          ('computed_at', 'timestamp', False)],
 'product_recommendations': [('product_id', 'long', False),
                             ('recommended_product_id', 'long', False),
                             ('support', 'double', False),
                             ('confidence', 'double', False),
                             ('lift', 'double', False),
                             ('computed_at', 'timestamp', False),
                             ('generated_at', 'timestamp', False),
                             ('model_run_id', 'string', False),
                             ('schema_version', 'string', False)]}


In [ ]:
# SHARED SOURCE (generated - do not edit): retail_setup/generation/job_locks.py
# Edit utility/src/retail_setup/generation/job_locks.py, then run utility/scripts/build_notebooks.py.

"""Lease locks that keep Lakehouse writers from overlapping.

``streaming-data-load`` (03, 04), ``daily-maintenance`` (05 and the ML
refresh), ``ml-required`` (setup-11 and the ML producers) and notebook 90 run
on independent schedules and write the same Delta tables. Fabric has no
cross-pipeline mutex, so each writer claims a lease row in
``<silver>._job_locks`` before it writes:

- ``silver``: notebook 03 and notebook 90 (all Silver tables and ``_watermarks``);
- ``gold``: notebook 04 and setup-11 (the ten Gold aggregates, ``setup_run_log``);
- ``ml:<table>``: the producer of that ML output table.

Notebook 05 claims the same lock for each table it maintains, without
waiting, and skips the table when another job holds it.

A claim is a Delta ``MERGE`` that succeeds only when the row is missing,
expired, or already ours; concurrent claims conflict in Delta and one of them
loses. The holder renews its lease from a background heartbeat, so a crashed
notebook releases its locks once the lease expires (``JOB_LOCK_TTL_SECONDS``).
All times come from Spark's ``current_timestamp()``, never the driver clock.

Keep this module free of package imports: Lakehouse notebooks run it as a
cell (``scripts/build_notebooks.py`` copies it in).
"""

import threading
import time
import uuid
from collections.abc import Callable, Iterable
from datetime import datetime, timedelta, timezone

JOB_LOCK_TABLE = "_job_locks"
JOB_LOCK_TTL_SECONDS = 900
JOB_LOCK_HEARTBEAT_SECONDS = 120
JOB_LOCK_POLL_SECONDS = 30
SILVER_LOCK = "silver"
GOLD_LOCK = "gold"


def ml_output_lock(table: str) -> str:
    """The lock that guards one ML output table."""
    return f"ml:{table}"


def is_concurrent_write_error(exc: BaseException) -> bool:
    """True for Delta optimistic-concurrency conflicts (another writer won)."""
    text = f"{type(exc).__name__}: {exc}"
    return "Concurrent" in text or "DELTA_CONCURRENT" in text


class DeltaJobLockStore:
    """Lease rows in a Delta table: (lock_name, owner, acquired_at, expires_at)."""

    def __init__(self, spark, table_name: str, ttl_seconds: int = JOB_LOCK_TTL_SECONDS):
        self.spark = spark
        self.table_name = table_name
        self.ttl_seconds = int(ttl_seconds)
        self.spark.sql(
            f"CREATE TABLE IF NOT EXISTS {table_name} "
            "(lock_name STRING, owner STRING, acquired_at TIMESTAMP, expires_at TIMESTAMP) "
            "USING DELTA"
        )

    @staticmethod
    def _quote(value: str) -> str:
        return "'" + value.replace("\\", "\\\\").replace("'", "\\'") + "'"

    def holder(self, name: str) -> str | None:
        """The owner of a live lease on ``name``, or None."""
        rows = self.spark.sql(
            f"SELECT owner FROM {self.table_name} "
            f"WHERE lock_name = {self._quote(name)} AND expires_at > current_timestamp()"
        ).collect()
        return rows[0]["owner"] if rows else None

    def try_claim(self, name: str, owner: str) -> bool:
        try:
            self.spark.sql(
                f"""
                MERGE INTO {self.table_name} AS t
                USING (
                    SELECT {self._quote(name)} AS lock_name, {self._quote(owner)} AS owner,
                           current_timestamp() AS acquired_at,
                           current_timestamp() + INTERVAL {self.ttl_seconds} SECONDS AS expires_at
                ) AS s
                ON t.lock_name = s.lock_name
                WHEN MATCHED AND (t.expires_at <= s.acquired_at OR t.owner = s.owner)
                    THEN UPDATE SET t.owner = s.owner, t.acquired_at = s.acquired_at,
                                    t.expires_at = s.expires_at
                WHEN NOT MATCHED THEN INSERT (lock_name, owner, acquired_at, expires_at)
                    VALUES (s.lock_name, s.owner, s.acquired_at, s.expires_at)
                """
            )
        except Exception as exc:
            if is_concurrent_write_error(exc):
                return False
            raise
        return self.holder(name) == owner

    def renew(self, name: str, owner: str) -> bool:
        try:
            self.spark.sql(
                f"UPDATE {self.table_name} "
                f"SET expires_at = current_timestamp() + INTERVAL {self.ttl_seconds} SECONDS "
                f"WHERE lock_name = {self._quote(name)} AND owner = {self._quote(owner)}"
            )
        except Exception as exc:
            if not is_concurrent_write_error(exc):
                raise
        return self.holder(name) == owner

    def release(self, name: str, owner: str) -> None:
        for _attempt in range(5):
            try:
                self.spark.sql(
                    f"DELETE FROM {self.table_name} "
                    f"WHERE lock_name = {self._quote(name)} AND owner = {self._quote(owner)}"
                )
                return
            except Exception as exc:
                if not is_concurrent_write_error(exc):
                    raise
                time.sleep(1)


class MemoryJobLockStore:
    """The same lease semantics in memory, with an injectable clock (tests)."""

    def __init__(
        self,
        ttl_seconds: int = JOB_LOCK_TTL_SECONDS,
        clock: Callable[[], datetime] | None = None,
    ):
        self.ttl = timedelta(seconds=ttl_seconds)
        self.clock = clock or (lambda: datetime.now(timezone.utc).replace(tzinfo=None))
        self.rows: dict[str, tuple[str, datetime]] = {}

    def holder(self, name: str) -> str | None:
        row = self.rows.get(name)
        return row[0] if row and row[1] > self.clock() else None

    def try_claim(self, name: str, owner: str) -> bool:
        now = self.clock()
        row = self.rows.get(name)
        if row is None or row[1] <= now or row[0] == owner:
            self.rows[name] = (owner, now + self.ttl)
        return self.holder(name) == owner

    def renew(self, name: str, owner: str) -> bool:
        row = self.rows.get(name)
        if row is not None and row[0] == owner:
            self.rows[name] = (owner, self.clock() + self.ttl)
        return self.holder(name) == owner

    def release(self, name: str, owner: str) -> None:
        if self.rows.get(name, ("", None))[0] == owner:
            del self.rows[name]


def new_lock_owner(holder: str) -> str:
    """A unique owner for one notebook run, e.g. ``03-streaming-to-silver:1a2b...``."""
    return f"{holder}:{uuid.uuid4().hex[:12]}"


def acquire_job_locks(
    store,
    names: Iterable[str],
    owner: str,
    *,
    wait_seconds: float,
    poll_seconds: float = JOB_LOCK_POLL_SECONDS,
    sleep: Callable[[float], None] = time.sleep,
    monotonic: Callable[[], float] = time.monotonic,
) -> list[str] | None:
    """Claim every lock in ``names`` or none of them.

    Locks are claimed in sorted order, so two jobs that need several locks
    can't deadlock. Retries until ``wait_seconds`` elapse (0 = one attempt)
    and returns the claimed names, or None when another job still holds one.
    """
    ordered = sorted(set(names))
    deadline = monotonic() + max(wait_seconds, 0)
    while True:
        claimed = []
        for name in ordered:
            if not store.try_claim(name, owner):
                break
            claimed.append(name)
        if len(claimed) == len(ordered):
            return claimed
        for name in reversed(claimed):
            store.release(name, owner)
        remaining = deadline - monotonic()
        if remaining <= 0:
            return None
        sleep(min(poll_seconds, remaining))


class JobLocks:
    """A notebook's set of leases, renewed by a heartbeat thread until released."""

    def __init__(
        self,
        store,
        holder: str,
        names: Iterable[str],
        heartbeat_seconds: float = JOB_LOCK_HEARTBEAT_SECONDS,
    ):
        self.store = store
        self.owner = new_lock_owner(holder)
        self.names = sorted(set(names))
        self.heartbeat_seconds = heartbeat_seconds
        self.held: list[str] = []
        self.lost: list[str] = []
        self._stop = threading.Event()
        self._thread: threading.Thread | None = None

    def blockers(self) -> dict[str, str | None]:
        """Current holders of the locks this job needs (for messages)."""
        return {name: self.store.holder(name) for name in self.names}

    def acquire(self, wait_seconds: float, **kwargs) -> bool:
        claimed = acquire_job_locks(
            self.store, self.names, self.owner, wait_seconds=wait_seconds, **kwargs
        )
        if claimed is None:
            return False
        self.held = claimed
        if self.heartbeat_seconds > 0:
            self._thread = threading.Thread(target=self._heartbeat, daemon=True)
            self._thread.start()
        return True

    def _heartbeat(self) -> None:
        while not self._stop.wait(self.heartbeat_seconds):
            self.renew()

    def renew(self) -> None:
        for name in self.held:
            try:
                renewed = self.store.renew(name, self.owner)
            except Exception as exc:
                print(f"Lock {name}: renewal failed ({exc}); retrying at the next heartbeat")
                continue
            if not renewed and name not in self.lost:
                self.lost.append(name)

    def assert_held(self) -> None:
        """Fail before a commit point if a lease expired and another job took it."""
        if self.lost:
            raise RuntimeError(f"Job locks lost before completion: {self.lost}")

    def release(self) -> None:
        self._stop.set()
        if self._thread is not None:
            self._thread.join(timeout=self.heartbeat_seconds + 5)
        for name in reversed(self.held):
            self.store.release(name, self.owner)
        self.held = []

In [ ]:
# One writer per ML output: this notebook runs in more than one pipeline, and
# notebook 05 maintains the same table (see job_locks).
LOCK_WAIT_SECONDS = int(get_env("LOCK_WAIT_SECONDS", default="1800"))
JOB_LOCKS = JobLocks(
    DeltaJobLockStore(spark, f"{LAKEHOUSE_NAME}.{SILVER_DB}.{JOB_LOCK_TABLE}"),
    "07-ml-market-basket",
    [ml_output_lock("product_associations"), ml_output_lock("product_recommendations")],
)
if not JOB_LOCKS.acquire(wait_seconds=LOCK_WAIT_SECONDS):
    raise RuntimeError(f"ML output locked by another job: {JOB_LOCKS.blockers()}")

In [ ]:
# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def ensure_database(name):
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {LAKEHOUSE_NAME}.{name}")

def read_silver(table_name):
    return spark.table(f"{LAKEHOUSE_NAME}.{SILVER_DB}.{table_name}")

def save_gold(df, table_name):
    full_name = f"{LAKEHOUSE_NAME}.{GOLD_DB}.{table_name}"
    df.write.format("delta").mode("overwrite").option(
        "overwriteSchema", "true"
    ).saveAsTable(full_name)
    row_count = spark.table(full_name).count()
    print(f"  {full_name}: {row_count} rows")
    return full_name, row_count

def silver_exists(table_name):
    try:
        spark.table(f"{LAKEHOUSE_NAME}.{SILVER_DB}.{table_name}")
        return True
    except AnalysisException:
        return False

def canonicalize_itemset(column):
    return F.sort_array(F.array_distinct(column))

def sample_baskets(baskets, total_baskets, max_baskets):
    """A deterministic hash sample of at most about ``max_baskets`` baskets."""
    if total_baskets <= max_baskets:
        return baskets
    keep_per_million = int(1_000_000 * max_baskets / total_baskets)
    return baskets.filter(
        F.pmod(F.xxhash64("receipt_id_ext"), F.lit(1_000_000)) < F.lit(keep_per_million)
    )

def cooccurrence_rules(baskets, total_baskets, min_pair_baskets):
    """Single-item rules for pairs bought together in >= min_pair_baskets baskets.

    ``baskets`` holds every basket (single-item ones too) and ``total_baskets``
    is their count, so support is the share of all baskets containing both
    products, confidence is P(consequent | antecedent), and lift is confidence
    divided by the consequent's basket share. Products bought in fewer than
    ``min_pair_baskets`` baskets can't form a qualifying pair and are dropped
    before the pair self-join.
    """
    items = baskets.select("receipt_id_ext", F.explode("items").alias("product_id"))
    item_frequency = items.groupBy("product_id").agg(F.count("*").alias("item_freq"))
    frequent_items = items.join(
        item_frequency.filter(F.col("item_freq") >= F.lit(min_pair_baskets)).select("product_id"),
        "product_id",
        "left_semi",
    )
    unordered_pairs = (
        frequent_items.alias("a")
        .join(frequent_items.alias("b"), F.col("a.receipt_id_ext") == F.col("b.receipt_id_ext"))
        .filter(F.col("a.product_id") < F.col("b.product_id"))
        .groupBy(F.col("a.product_id").alias("first_id"), F.col("b.product_id").alias("second_id"))
        .agg(F.count("*").alias("pair_freq"))
        .filter(F.col("pair_freq") >= F.lit(min_pair_baskets))
    )
    pairs = unordered_pairs.select(
        F.col("first_id").alias("antecedent_id"),
        F.col("second_id").alias("consequent_id"),
        "pair_freq",
    ).unionByName(
        unordered_pairs.select(
            F.col("second_id").alias("antecedent_id"),
            F.col("first_id").alias("consequent_id"),
            "pair_freq",
        )
    )
    antecedent_frequency = item_frequency.select(
        F.col("product_id").alias("antecedent_id"), F.col("item_freq").alias("antecedent_freq")
    )
    consequent_frequency = item_frequency.select(
        F.col("product_id").alias("consequent_id"), F.col("item_freq").alias("consequent_freq")
    )
    total = F.lit(float(total_baskets))
    return (
        pairs
        .join(antecedent_frequency, "antecedent_id")
        .join(consequent_frequency, "consequent_id")
        .withColumn("confidence", F.col("pair_freq") / F.col("antecedent_freq"))
        .select(
            F.array(F.col("antecedent_id").cast("long")).alias("antecedent"),
            F.array(F.col("consequent_id").cast("long")).alias("consequent"),
            (F.col("pair_freq") / total).alias("support"),
            F.col("confidence").cast("double").alias("confidence"),
            (F.col("confidence") / (F.col("consequent_freq") / total)).alias("lift"),
        )
    )

def select_rules(rules, min_support, min_confidence, min_lift, per_product, top_n):
    """Rules above the thresholds: the best ``per_product`` per antecedent by
    lift, then the ``top_n`` best overall."""
    per_antecedent = Window.partitionBy("antecedent").orderBy(
        F.desc("lift"), F.desc("support"), "consequent"
    )
    return (
        rules.filter(
            (F.col("support") >= F.lit(min_support))
            & (F.col("confidence") >= F.lit(min_confidence))
            & (F.col("lift") > F.lit(min_lift))
        )
        .withColumn("_rank", F.row_number().over(per_antecedent))
        .filter(F.col("_rank") <= F.lit(per_product))
        .drop("_rank")
        .orderBy(F.desc("lift"), F.desc("support"), "antecedent", "consequent")
        .limit(top_n)
    )

ensure_database(GOLD_DB)

mlflow.set_experiment(EXPERIMENT_NAME)

def _normalize_ml_type(data_type):
    value = data_type.replace("bigint", "long").replace("integer", "int")
    return value


def validate_ml_output(frame, table_name):
    contract = ML_OUTPUT_CONTRACTS[table_name]
    expected = tuple((name, data_type) for name, data_type, _ in contract)
    actual = tuple(
        (field.name, _normalize_ml_type(field.dataType.simpleString()))
        for field in frame.schema.fields
    )
    if actual != expected:
        raise RuntimeError(
            f"ML output schema mismatch for {table_name}: expected={expected}, actual={actual}"
        )
    non_nullable = tuple(name for name, _, nullable in contract if not nullable)
    null_flags = frame.agg(
        *(F.max(F.col(name).isNull().cast("int")).alias(name) for name in non_nullable)
    ).first().asDict()
    null_columns = [name for name in non_nullable if null_flags.get(name)]
    if null_columns:
        raise RuntimeError(
            f"ML output {table_name} has nulls in required columns: {null_columns}"
        )
    return frame



In [ ]:
print("="*60)
print("MARKET BASKET ANALYSIS - TRANSACTION PREPARATION")
print("="*60)

In [ ]:
# Step 1: Prepare transaction baskets: every SALE receipt's distinct products.

if not silver_exists(RECEIPTS_TABLE) or not silver_exists(RECEIPT_LINES_TABLE):
    raise RuntimeError(
        f"Required tables {RECEIPTS_TABLE} and {RECEIPT_LINES_TABLE} not found in Silver"
    )

print("\nPreparing transaction baskets...")

all_receipts = read_silver(RECEIPTS_TABLE)
# One source as-of for every required ML output: the latest Silver receipt.
source_as_of = all_receipts.agg(
    F.max(F.col("event_ts").cast("timestamp")).alias("source_as_of")
).first()["source_as_of"]
if source_as_of is None:
    raise RuntimeError(f"{RECEIPTS_TABLE} contains no source timestamps.")
receipts = (
    all_receipts
    .filter(F.col("receipt_type") == RECEIPT_TYPE_FILTER)
    .filter(F.col("event_ts").cast("timestamp") <= F.lit(source_as_of))
    .select("receipt_id_ext")
)
receipt_lines = read_silver(RECEIPT_LINES_TABLE).select("receipt_id_ext", "product_id")

all_baskets = (
    receipts
    .join(receipt_lines, "receipt_id_ext")
    .groupBy("receipt_id_ext")
    .agg(canonicalize_itemset(F.collect_set("product_id")).alias("items"))
)
all_basket_count = all_baskets.count()
baskets = sample_baskets(all_baskets, all_basket_count, MAX_PAIR_BASKETS).cache()
total_baskets = baskets.count()
multi_item_baskets = baskets.filter(F.size(F.col("items")) > 1).count()
print(f"  Source as-of: {source_as_of}")
print(f"  SALE baskets: {all_basket_count:,}; analyzed: {total_baskets:,}")
print(f"  Multi-item baskets analyzed: {multi_item_baskets:,}")

if multi_item_baskets == 0:
    print("  No multi-item transactions found; empty outputs will replace prior results.")

In [ ]:
print("\n" + "="*60)
print("PAIR CO-OCCURRENCE RULES")
print("="*60)

frequent_itemsets_count = 0
candidate_rules = None
if multi_item_baskets > 0:
    candidate_rules = cooccurrence_rules(baskets, total_baskets, MIN_PAIR_BASKETS).cache()
    frequent_itemsets_count = candidate_rules.count() // 2
    print(f"\nPairs bought together in >= {MIN_PAIR_BASKETS} baskets: {frequent_itemsets_count:,}")
else:
    print("\nSkipping rule mining because there are no multi-item baskets.")

In [ ]:
print("\n" + "="*60)
print("ASSOCIATION RULE SELECTION")
print("="*60)

from pyspark.sql.types import ArrayType, DoubleType, LongType, StructField, StructType, TimestampType

result_schema = StructType([
    StructField("antecedent", ArrayType(LongType()), False),
    StructField("consequent", ArrayType(LongType()), False),
    StructField("support", DoubleType(), False),
    StructField("confidence", DoubleType(), False),
    StructField("lift", DoubleType(), False),
    StructField("computed_at", TimestampType(), False),
])

total_rules = 0
RULE_SOURCE = "cooccurrence"
result_df = spark.createDataFrame([], result_schema)

if candidate_rules is not None:
    selected = select_rules(
        candidate_rules, MIN_SUPPORT, MIN_CONFIDENCE, MIN_LIFT, MAX_RULES_PER_PRODUCT, TOP_N_RULES
    )
    total_rules = candidate_rules.filter(
        (F.col("support") >= F.lit(MIN_SUPPORT))
        & (F.col("confidence") >= F.lit(MIN_CONFIDENCE))
        & (F.col("lift") > F.lit(MIN_LIFT))
    ).count()
    print(f"Rules meeting the thresholds: {total_rules:,}")
    if total_rules == 0:
        fallback_rules = cooccurrence_rules(baskets, total_baskets, FALLBACK_MIN_PAIR_BASKETS)
        fallback_count = fallback_rules.count()
        print(
            "\nNo rule met the configured thresholds; using the "
            f"{fallback_count:,} product pairs bought together in at least "
            f"{FALLBACK_MIN_PAIR_BASKETS} baskets."
        )
        if fallback_count > 0:
            RULE_SOURCE = "cooccurrence_fallback"
            total_rules = fallback_count
            selected = select_rules(
                fallback_rules, 0.0, 0.0, 0.0, MAX_RULES_PER_PRODUCT, TOP_N_RULES
            )
        else:
            selected = None
            print("No product pair repeats; publishing empty outputs.")
    if selected is not None:
        result_df = selected.withColumn("computed_at", F.lit(source_as_of).cast("timestamp"))
        print(f"\nTop {TOP_N_RULES} association rules by lift prepared")
else:
    print("\nNo association rules; publishing empty outputs.")

In [ ]:
print("\n" + "="*60)
print("SAVING TO GOLD LAYER")
print("="*60)

# Save association rules to the configured gold table
result_df = validate_ml_output(result_df, "product_associations")
rules_table, rules_saved_count = save_gold(result_df, PRODUCT_ASSOCIATIONS_TABLE)
saved_rules_df = spark.table(rules_table)

print("\nSample association rules (top 10 by lift):")
if saved_rules_df.limit(1).count() > 0:
    saved_rules_df.orderBy(F.desc("lift")).limit(10).show(truncate=False)
else:
    print("  No rules to display")

In [ ]:
# Log results to MLflow
with mlflow.start_run(
    run_name=f"basket_pairs_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M')}"
) as run:
    mlflow.log_params({
        "algorithm": "pair_cooccurrence",
        "min_support": MIN_SUPPORT,
        "min_confidence": MIN_CONFIDENCE,
        "min_lift": MIN_LIFT,
        "top_n_rules": TOP_N_RULES,
        "max_rules_per_product": MAX_RULES_PER_PRODUCT,
        "rule_source": RULE_SOURCE,
        "min_pair_baskets": MIN_PAIR_BASKETS,
        "fallback_min_pair_baskets": FALLBACK_MIN_PAIR_BASKETS,
        "sale_baskets": all_basket_count,
        "analyzed_baskets": total_baskets,
        "source_as_of": source_as_of.isoformat(),
    })

    rules_count = spark.table(PRODUCT_ASSOCIATIONS_TABLE_NAME).count()
    mlflow.log_metrics({
        "rules_saved": rules_count,
    })

    print(f"MLflow run: {run.info.run_id}")
    print(f"Rules logged: {rules_count}")


In [ ]:
print("\n" + "="*60)
print("CREATING PRODUCT RECOMMENDATION TABLE")
print("="*60)

# product_recommendations is a required-tier ML output, so each row carries the
# MLflow run that produced it. The run is opened above, before this write, and
# `run` stays bound after the block so the run id is available here.
rules_df = spark.table(PRODUCT_ASSOCIATIONS_TABLE_NAME)
recommendations = (
    rules_df
    .withColumn("antecedent_product", F.explode("antecedent"))
    .withColumn("consequent_product", F.explode("consequent"))
    .select(
        F.col("antecedent_product").alias("product_id"),
        F.col("consequent_product").alias("recommended_product_id"),
        "support",
        "confidence",
        "lift",
        "computed_at",
    )
)

# Singleton rules map one-to-one to recommendation pairs, so every metric
# remains from the same mined rule.
recommendations = recommendations.orderBy(
    "product_id", "recommended_product_id"
)
recommendations = (
    recommendations
    .withColumn("generated_at", F.current_timestamp())
    .withColumn("model_run_id", F.lit(run.info.run_id).cast("string"))
    .withColumn("schema_version", F.lit(ML_SCHEMA_VERSION).cast("string"))
)
recommendations = validate_ml_output(recommendations, "product_recommendations")

recommendations_table, recommendations_count = save_gold(
    recommendations,
    PRODUCT_RECOMMENDATIONS_TABLE,
)

if recommendations_count > 0:
    print("\nSample recommendations (top 10):")
    (
        spark.table(recommendations_table)
        .orderBy("product_id", F.desc("lift"))
        .limit(10)
        .show(truncate=False)
    )
else:
    print("  No recommendations to display; empty table replaced any prior results.")


In [ ]:
print("\n" + "="*60)
print("MARKET BASKET ANALYSIS COMPLETE")
print("="*60)

# Summary statistics
rules_saved_count = spark.table(PRODUCT_ASSOCIATIONS_TABLE_NAME).count()
recommendations_count = (
    spark.table(PRODUCT_RECOMMENDATIONS_TABLE_NAME).count()
    if spark.catalog.tableExists(PRODUCT_RECOMMENDATIONS_TABLE_NAME)
    else 0
)
print(f"\nSummary:")
print(f"  Transaction baskets analyzed: {total_baskets:,}")
print(f"  Frequent pairs: {frequent_itemsets_count:,}")
print(f"  Association rules (all): {total_rules:,}")
print(f"  Top rules saved: {rules_saved_count}")
print(f"  Product recommendations saved: {recommendations_count}")

# Show Gold tables
gold_tables = spark.sql(f"SHOW TABLES IN {LAKEHOUSE_NAME}.{GOLD_DB}").collect()
print(f"\nGold ({GOLD_DB}): {len(gold_tables)} tables")

In [ ]:
# Release the ML output lock (see job_locks).
JOB_LOCKS.assert_held()
JOB_LOCKS.release()